In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, r2_score

print("--- 1. Initializing ML Pipeline with GDP Clusters ---")
df = pd.read_csv('Enriched_Flight_Visa_Data.csv')
df_ml = df.dropna(subset=['GDP_pc', 'Population', 'Year', 'Visa_Acceptance_Rate', 'Outbound_Per_Capita', 'GDP_Cluster'])

# One-Hot Encode the GDP_Cluster column so the model can understand the 4 tiers
df_ml = pd.get_dummies(df_ml, columns=['GDP_Cluster'], drop_first=False)

# Define Features (including the newly created cluster columns) and Target
features = [
    'Population', 'Year', 'Visa_Acceptance_Rate',
    'GDP_Cluster_Highest 25%', 'GDP_Cluster_Lower High 50-75%',
    'GDP_Cluster_Higher Low 25-50%', 'GDP_Cluster_Lowest 25%'
]
X = df_ml[features]
y = df_ml['Outbound_Per_Capita']

# Scale Features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

print("Training Models...")
baseline_y_pred = np.full(len(y_test), y_train.mean())
lr = LinearRegression().fit(X_train, y_train)
knn = GridSearchCV(KNeighborsRegressor(), {'n_neighbors': [3, 5, 7, 10]}, cv=5).fit(X_train, y_train).best_estimator_
rf = GridSearchCV(RandomForestRegressor(random_state=42), {'n_estimators': [100, 200], 'max_depth': [5, 10, None]}, cv=5).fit(X_train, y_train).best_estimator_

# Evaluation
models = {"Linear Regression": lr, "KNN (Tuned)": knn, "Random Forest (Tuned)": rf}
results = [{"Model": "Mean Baseline", "R2": r2_score(y_test, baseline_y_pred), "RMSE": np.sqrt(mean_squared_error(y_test, baseline_y_pred))}]

for name, model in models.items():
    y_pred = model.predict(X_test)
    results.append({"Model": name, "R2": r2_score(y_test, y_pred), "RMSE": np.sqrt(mean_squared_error(y_test, y_pred))})

results_df = pd.DataFrame(results)
print("\n--- Model Performance Summary ---")
print(results_df)

# Visualizations
sns.set_theme(style="whitegrid")

# GRAPH 3: Model Comparison
plt.figure(figsize=(10, 6))
sns.barplot(x="Model", y="R2", data=results_df, palette="magma")
plt.axhline(0, color='black', lw=1)
plt.title("R-Squared Score Comparison", fontsize=14)
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig('ml_model_comparison.png')
plt.show()

# GRAPH 4: Feature Importance (Evaluating the impact of specific clusters)
plt.figure(figsize=(10, 6))
importances = rf.feature_importances_
# Clean up feature names for the chart
clean_features = ['Population', 'Year', 'Visa Acceptance Rate', 'Tier: Highest 25%', 'Tier: Lower High', 'Tier: Higher Low', 'Tier: Lowest 25%']

sns.barplot(x=importances, y=clean_features, palette="viridis")
plt.title("Feature Importance: How do specific Economic Tiers drive traffic?", fontsize=14)
plt.xlabel("Importance Weight")
plt.tight_layout()
plt.savefig('ml_feature_importance_clusters.png')
plt.show()